<a href="https://colab.research.google.com/github/JordanTerwilliger/Intro-to-Deep-Learning/blob/main/HW1/HW1_CIFAR10_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Jordan Terwilliger, 801343938, HW1

https://github.com/JordanTerwilliger/HW1-CIFAR10-CNN

In [2]:
import numpy as np

import torch as torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt

In [3]:
torch.manual_seed(1)

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

train_data = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle = True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle = True, num_workers=2)

100%|██████████| 170M/170M [00:13<00:00, 12.5MB/s]


In [5]:
class_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'] # CIFAR-10 image classes

Alex Net consists of 8 Layers with a process of:

Input -> Conv2d -> Pooling -> Conv2d -> Pooling -> Conv2d -> Conv2d -> Conv2d -> Pooling -> Linear -> Linear -> Output

Using CIFAR-10 with this means that our pooling layers need a reduced stride or the layer can be removed altogether. The Conv2d layers can stay with 3x3 kernels with a reduced stride. We dont need 4096 neurons in the Linear layers, 128 or even 64 may suffice. LeNet which has 28x28 images uses 120 and then 84 for the Hidden Layers.

In [6]:
class ModifiedAlex(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
    self.pool = nn.MaxPool2d(2,2)
    self.conv2 = nn.Conv2d(16,16,3, padding =1)
    self.fc1 = nn.Linear(16*8*8,120)
    self.fc2 = nn.Linear(120,10)

  def forward(self,x):
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = torch.flatten(x,1)
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x

In [12]:
net = ModifiedAlex()
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr = 0.001, momentum = 0.9)

In [13]:
epochs = 30
#Lists for storing loss and validation values
train_loss_list = []
val_loss_list = []
val_accuracy_list = []
for epoch in range(epochs):
  running_loss = 0.0
  net.train()
  for i, data in enumerate(train_loader):
    inputs, labels = data
    inputs = inputs
    labels = labels

    optimizer.zero_grad()

    outputs = net(inputs)

    loss_function = loss(outputs, labels)
    loss_function.backward()
    optimizer.step()

    running_loss += loss_function.item()

  correct = 0
  total = 0

  net.eval()
  with torch.no_grad():
    for data in test_loader:
      images, labels = data
      outputs = net(images)
      loss_function = loss(outputs, labels)
      running_loss += loss_function.item()
      _, predicted = torch.max(outputs.data, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()
  val_loss_list.append(running_loss / len(test_loader))
  val_accuracy = 100 * correct / total
  val_accuracy_list.append(val_accuracy)

  train_loss_list.append(running_loss/len(train_loader))
  print(f'Epoch: {epoch}, Loss: {running_loss / len(train_loader):.4f}, Validation Accuracy: {val_accuracy}')

Epoch: 0, Loss: 2.3606, Accuracy: 39.25
Epoch: 1, Loss: 1.8677, Accuracy: 46.01
Epoch: 2, Loss: 1.6900, Accuracy: 49.67
Epoch: 3, Loss: 1.5568, Accuracy: 55.0
Epoch: 4, Loss: 1.4579, Accuracy: 57.3
Epoch: 5, Loss: 1.3832, Accuracy: 59.24
Epoch: 6, Loss: 1.3296, Accuracy: 59.86
Epoch: 7, Loss: 1.2683, Accuracy: 62.15
Epoch: 8, Loss: 1.2204, Accuracy: 61.57
Epoch: 9, Loss: 1.1681, Accuracy: 63.41
Epoch: 10, Loss: 1.1157, Accuracy: 65.17
Epoch: 11, Loss: 1.0764, Accuracy: 64.68
Epoch: 12, Loss: 1.0387, Accuracy: 65.51
Epoch: 13, Loss: 1.0004, Accuracy: 66.24
Epoch: 14, Loss: 0.9696, Accuracy: 66.1
Epoch: 15, Loss: 0.9288, Accuracy: 67.33
Epoch: 16, Loss: 0.8983, Accuracy: 67.64
Epoch: 17, Loss: 0.8685, Accuracy: 67.76
Epoch: 18, Loss: 0.8463, Accuracy: 67.21
Epoch: 19, Loss: 0.8184, Accuracy: 67.56
Epoch: 20, Loss: 0.7925, Accuracy: 67.74
Epoch: 21, Loss: 0.7666, Accuracy: 67.77
Epoch: 22, Loss: 0.7422, Accuracy: 68.07
Epoch: 23, Loss: 0.7220, Accuracy: 67.78
Epoch: 24, Loss: 0.6968, Accu